# 1. Kiểm tra GPU
Notebook này nên chạy trên Google Colab T4/A100.


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f'VRAM free: {free/1e9:.2f} GB / total: {total/1e9:.2f} GB')
else:
    print('NO GPU - chỉ nên dùng để setup hoặc test rất ngắn')


# 2. Mount Google Drive tùy chọn
Dùng nếu muốn cache model qua nhiều session.


In [ ]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'


# 3. Clone repo và cài dependencies


In [ ]:
%cd /content
!rm -rf Fast-VietTTS
!git clone https://github.com/baobui0709/Fast-VietTTS.git
%cd Fast-VietTTS

!pip install -q --upgrade pip
!pip install -q --index-url https://download.pytorch.org/whl/cu124 torch==2.6.0 torchaudio==2.6.0
!pip install -q numpy==1.26.4 librosa soundfile num2words tqdm huggingface_hub gradio
!pip install -q s3tokenizer omegaconf conformer safetensors
!pip uninstall -y -q torchvision torchtext || true

%cd /content
!rm -rf TienHiep-TTS
!git clone https://github.com/yingchunbin/TienHiep-TTS.git
%cd /content/Fast-VietTTS


# 4. Import helper chung


In [ ]:
import sys, os, time, torch, torchaudio
from IPython.display import Audio, display

viterbox_src = '/content/TienHiep-TTS'
if viterbox_src not in sys.path:
    sys.path.insert(0, viterbox_src)

from src.engine import FastVietTTS
from src.audio_utils import prepare_reference, get_audio_duration


# 5. Load model


In [ ]:
tts = FastVietTTS(device='cuda' if torch.cuda.is_available() else 'cpu', use_fp16=False, compile_model=False)
print('Model loaded')


# 6. Generate câu ngắn không cần reference audio


In [ ]:
text = 'Xin chào! Đây là kiểm tra Fast-VietTTS trên Google Colab.'
t0 = time.time()
audio = tts.generate(text, language='vi')
elapsed = time.time() - t0
duration = audio.shape[-1] / 24000
print(f'Elapsed: {elapsed:.2f}s | Audio: {duration:.2f}s | RTF: {elapsed/max(duration,1e-6):.2f}x')
tts.save_audio(audio, '/content/quickstart_output.wav')
display(Audio('/content/quickstart_output.wav'))


# 7. Upload reference audio và voice clone


In [ ]:
from google.colab import files
uploaded = files.upload()
ref_path = next(iter(uploaded.keys())) if uploaded else None
if ref_path:
    processed = prepare_reference(ref_path, output_path='/content/ref_processed.wav')
    audio2 = tts.generate('Tôi đang thử clone giọng nói từ file mẫu.', audio_prompt=processed, language='vi')
    tts.save_audio(audio2, '/content/voice_clone_output.wav')
    display(Audio('/content/voice_clone_output.wav'))


# 8. Download audio


In [ ]:
from google.colab import files
files.download('/content/quickstart_output.wav')
if os.path.exists('/content/voice_clone_output.wav'):
    files.download('/content/voice_clone_output.wav')
print('Tóm tắt: QuickStart hoàn tất')
